# Week 37, Day 1: Exception hierarchies

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w037d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
for err in [FileNotFoundError("x"), KeyError("k"), ZeroDivisionError(), KeyboardInterrupt()]:
    try:
        raise err
    except LookupError:
        print(type(err).__name__, "-> a lookup problem")
    except OSError:
        print(type(err).__name__, "-> a problem with the system")
    except Exception:
        print(type(err).__name__, "-> some other problem")
    except BaseException:
        print(type(err).__name__, "-> not a problem with the program at all")

In [ ]:
for cls in [FileNotFoundError, UnicodeDecodeError, RecursionError, TimeoutError, KeyError]:
    chain = [c.__name__ for c in cls.__mro__[1:-1]]
    print(f"{cls.__name__:<20} is a {' -> '.join(chain)}")
print(issubclass(PermissionError, OSError), issubclass(KeyError, Exception), issubclass(KeyboardInterrupt, Exception))

In [ ]:
def read_setting(settings, path):
    try:
        with open(path) as f:
            return settings[f.read().strip()]
    except FileNotFoundError:
        return f"no file {path}"
    except OSError as err:
        return f"couldn't read {path}: {type(err).__name__}"
    except LookupError:
        return "unknown setting"

with open("which.txt", "w") as f:
    f.write("colour")
print(read_setting({"colour": "blue"}, "which.txt"))
print(read_setting({}, "which.txt"))
print(read_setting({}, "missing.txt"))
print(read_setting({}, "."))            # a folder: IsADirectoryError, a kind of OSError

In [ ]:
def attempt(text, log):
    try:
        n = int(text)
    except ValueError:
        log.append("bad number")
        return None
    else:
        log.append("ok")
        return n * 2
    finally:
        log.append("done")

for text in ["21", "x"]:
    log = []
    print(attempt(text, log), log)

In [ ]:
import sys

def risky(kind):
    if kind == "bug":
        raise ValueError("oops")
    if kind == "quit":
        sys.exit(3)

for kind in ["bug", "quit"]:
    try:
        try:
            risky(kind)
        except Exception as err:
            print("handled", type(err).__name__)
    except SystemExit as stop:
        print("sys.exit got through except Exception, with code", stop.code)

In [ ]:
def check_rows(rows):
    problems = []
    for i, row in enumerate(rows, start=1):
        if "name" not in row:
            problems.append(KeyError(f"row {i}: no name"))
        elif not str(row.get("age", "")).isdigit():
            problems.append(ValueError(f"row {i}: bad age {row.get('age')!r}"))
    if problems:
        raise ExceptionGroup("rows failed", problems)

rows = [{"name": "Ada", "age": "31"}, {"age": "40"}, {"name": "Bo", "age": "old"}, {"name": "Chi"}]
try:
    check_rows(rows)
except* KeyError as group:
    print("missing:", [str(e) for e in group.exceptions])
except* ValueError as group:
    print("invalid:", [str(e) for e in group.exceptions])

> **Key idea.** Python's exceptions form a tree: `BaseException` at the top, with `KeyboardInterrupt` and `SystemExit` directly under it, and `Exception` holding everything a program might handle, in families such as `LookupError`, `ArithmeticError`, `OSError` and `ValueError`. `except` catches a class and all its subclasses, so order clauses from specific to general, and catch only what you can handle there. `try` runs first, `except` on a matching error, `else` only if nothing was raised, and `finally` always; keep `try` small and never return from `finally`. Never use a bare `except`. `ExceptionGroup` carries several errors, and `except*` handles each type.

### Exercise: Which family?

Write `family(exc)`, which returns the family of an exception object: `"lookup"` for `LookupError`s, `"arithmetic"` for `ArithmeticError`s, `"system"` for `OSError`s, `"value"` for `ValueError`s, `"stop"` for anything that isn't an `Exception` at all (like `KeyboardInterrupt`), and `"other"` for any other `Exception`. Use `isinstance`, checked in a sensible order.

In [ ]:
def family(exc):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_family():\n    \"families\"\n    cases = [(KeyError(\"k\"), \"lookup\"), (IndexError(), \"lookup\"), (ZeroDivisionError(), \"arithmetic\"), (FileNotFoundError(), \"system\"),\n             (TimeoutError(), \"system\"), (UnicodeDecodeError(\"utf-8\", b\"x\", 0, 1, \"bad\"), \"value\"), (ValueError(), \"value\"),\n             (KeyboardInterrupt(), \"stop\"), (SystemExit(), \"stop\"), (TypeError(), \"other\"), (RecursionError(), \"other\")]\n    for exc, want in cases:\n        assert family(exc) == want, f\"{type(exc).__name__}: got {family(exc)!r}, expected {want!r}\"")

### Exercise: Every part of try

Write `load_age(text, log)`, which converts the text to an int with `int(text)` inside a `try` containing **only** that line. On a `ValueError`, it appends `"bad: "` and the text to `log` and returns `None`. If the conversion worked, an `else` part appends `"ok"` and returns the age, but raises `ValueError("too old")` for ages over 130 (an error from the `else` part isn't caught by the `except` part). Whatever happens, a `finally` part appends `"checked"`.

In [ ]:
def load_age(text, log):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parts():\n    \"try, except, else, finally\"\n    log = []\n    assert load_age(\"31\", log) == 31 and log == [\"ok\", \"checked\"]\n    log = []\n    assert load_age(\"x\", log) is None and log == [\"bad: x\", \"checked\"]\n    log = []\n    try:\n        load_age(\"200\", log)\n        assert False, \"too old should raise ValueError\"\n    except ValueError as err:\n        assert str(err) == \"too old\"\n    assert log == [\"ok\", \"checked\"], \"finally runs even when else raises\"\n    assert \"else:\" in source and \"finally:\" in source")

### Exercise: Trying each source

Write `first_success(sources)`, where `sources` is a list of functions taking no arguments. It calls them in order and returns the first result that doesn't raise an `Exception`. If they all fail, it raises an `ExceptionGroup` with the message `"all sources failed"` holding every error, in order. An empty list raises `ValueError`. Exceptions that aren't `Exception`s (like `KeyboardInterrupt`) must not be caught.

In [ ]:
def first_success(sources):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fallbacks():\n    \"fallbacks\"\n    def down():\n        raise ConnectionError(\"down\")\n    def bad():\n        raise ValueError(\"garbled\")\n    assert first_success([down, lambda: 42, bad]) == 42 and first_success([lambda: None]) is None\n    try:\n        first_success([down, bad])\n        assert False\n    except* ConnectionError as g:\n        assert [str(e) for e in g.exceptions] == [\"down\"]\n    except* ValueError as g:\n        assert [str(e) for e in g.exceptions] == [\"garbled\"]\n    try:\n        first_success([down, bad])\n    except ExceptionGroup as g:\n        assert str(g).startswith(\"all sources failed\") and len(g.exceptions) == 2\n    def stop():\n        raise KeyboardInterrupt\n    try:\n        first_success([stop, lambda: 1])\n        assert False, \"KeyboardInterrupt must get through\"\n    except KeyboardInterrupt:\n        pass\n    try:\n        first_success([])\n        assert False\n    except ValueError:\n        pass")

### Exercise: Counting what went wrong

Exception groups can contain other groups. Write `leaves(group)`, a list of all the plain exceptions inside a group, however deeply nested, in order; and `counts(group)`, a dictionary from each exception's class name to how many there are.

In [ ]:
def leaves(group):
    ___

def counts(group):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_leaves():\n    \"nested groups\"\n    inner = ExceptionGroup(\"inner\", [KeyError(\"a\"), ValueError(\"b\")])\n    g = ExceptionGroup(\"outer\", [ValueError(\"c\"), inner, ExceptionGroup(\"deep\", [ExceptionGroup(\"deeper\", [TypeError(\"d\")])])])\n    assert [str(e) for e in leaves(g)] == [\"c\", \"'a'\", \"b\", \"d\"]\n    assert counts(g) == {\"ValueError\": 2, \"KeyError\": 1, \"TypeError\": 1}\n    assert counts(ExceptionGroup(\"x\", [OSError()])) == {\"OSError\": 1}")

### Bug hunt: An except that never runs

A missing file should give the message `no such file`, but the general `OSError` clause comes first and catches it, so this prints `system problem`. Fix the order so it prints `no such file`.

In [ ]:
def open_report(path):
    try:
        with open(path) as f:
            return f.read()
    except OSError:
        return "system problem"
    except FileNotFoundError:
        return "no such file"

print(open_report("missing-report.txt"))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"specific first\"\n    expect_output(\"no such file\")")

### Bug hunt: The error that vanished

`save` should let the `ValueError` through (after its clean-up), but the `return` inside `finally` silently swallows it, so this prints `None` and then `cleaned up`. Fix `save` so it prints `cleaned up` and then `error: amount must be positive`.

In [ ]:
log = []

def save(amount):
    try:
        if amount <= 0:
            raise ValueError("amount must be positive")
        return "saved"
    finally:
        log.append("cleaned up")
        return None

try:
    print(save(-5))
except ValueError as err:
    print(log[0])
    print("error:", err)
else:
    print(log[0])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no return in finally\"\n    expect_output(\"cleaned up\\nerror: amount must be positive\")\n    assert save(3) == \"saved\"")

### Bug hunt: Caught too narrowly

`lookup` should handle any missing item, whether the collection is a dictionary or a list, but it only catches `KeyError`, so the list's `IndexError` crashes the program. Catch the family both belong to, so it prints `rice` then `none` then `none`.

In [ ]:
def lookup(collection, key):
    try:
        return collection[key]
    except KeyError:
        return "none"

print(lookup({"a": "rice"}, "a"))
print(lookup({"a": "rice"}, "b"))
print(lookup(["rice"], 5))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the whole family\"\n    expect_output(\"rice\\nnone\\nnone\")\n    assert \"LookupError\" in source")

### Puzzle: A tidy try

Arrange the lines into a function that keeps the `try` small, reports a bad number, and always closes the connection. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
def f(x, log):
    try:
        log.append("try")
        if x == 0:
            raise ZeroDivisionError
        if x < 0:
            raise ValueError
    except ArithmeticError:
        log.append("arith")
    else:
        log.append("else")
    finally:
        log.append("finally")

a, b, c = [], [], []
f(1, a)
f(0, b)
try:
    f(-1, c)
except ValueError:
    c.append("outside")

p = ___   # a
q = ___   # b
r = ___   # c
s = ___   # issubclass(KeyboardInterrupt, Exception)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [\"try\", \"else\", \"finally\"] and q == [\"try\", \"arith\", \"finally\"], \"p and q\"\n    assert r == [\"try\", \"finally\", \"outside\"] and s is False")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a careful batch runner

Write `run_batch(jobs, cleanup, strict=False)`. `jobs` is a list of `(name, function)` pairs; each function takes no arguments. Run them in order and return a list of results, one per job that ran: `(name, "ok", value)` for success, or `(name, "error", "ValueError: bad")` (the exception's class name and message) for an `Exception`. Then:

- `cleanup()` must be called exactly **once**, whatever happens, after the jobs.
- An exception that isn't an `Exception` (such as `KeyboardInterrupt`, or the `Abort` class in the checks, which inherits from `BaseException`) stops the batch at once: cleanup still runs, and the exception goes through to the caller.
- With `strict=True`, if any job failed, raise an `ExceptionGroup` (message `"batch had failures"`) of the failures' exceptions, after cleanup, instead of returning.

In [ ]:
def run_batch(jobs, cleanup, strict=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("class Abort(BaseException):\n    pass\ndef make():\n    calls = []\n    jobs = [(\"one\", lambda: 1), (\"bad\", lambda: int(\"x\")), (\"three\", lambda: 3)]\n    return calls, jobs\ndef test_batch():\n    \"running a batch\"\n    calls, jobs = make()\n    got = run_batch(jobs, lambda: calls.append(\"clean\"))\n    assert got == [(\"one\", \"ok\", 1), (\"bad\", \"error\", \"ValueError: invalid literal for int() with base 10: 'x'\"), (\"three\", \"ok\", 3)], f\"got {got}\"\n    assert calls == [\"clean\"] and run_batch([], lambda: calls.append(\"c2\")) == [] and calls == [\"clean\", \"c2\"]\ndef test_strict():\n    \"strict mode\"\n    calls, jobs = make()\n    try:\n        run_batch(jobs, lambda: calls.append(\"clean\"), strict=True)\n        assert False, \"strict should raise\"\n    except* ValueError as g:\n        assert len(g.exceptions) == 1\n    assert calls == [\"clean\"]\n    assert run_batch([(\"a\", lambda: 1)], lambda: None, strict=True) == [(\"a\", \"ok\", 1)]\ndef test_abort():\n    \"stopping\"\n    calls = []\n    def stop():\n        raise Abort()\n    ran = []\n    try:\n        run_batch([(\"a\", lambda: ran.append(\"a\")), (\"stop\", stop), (\"c\", lambda: ran.append(\"c\"))], lambda: calls.append(\"clean\"))\n        assert False, \"Abort should go through\"\n    except Abort:\n        pass\n    assert ran == [\"a\"] and calls == [\"clean\"]")

## Warm-up

A `FileNotFoundError` was caught as an `OSError`, a `KeyError` as a `LookupError`, and a `KeyboardInterrupt` (what pressing Ctrl+C raises) slipped past `except Exception` entirely. Each `except` catches a whole **branch** of a tree. Today maps that tree, so you always know what you're catching.

## The exception tree

Every exception is a class, and they're arranged by inheritance (Week 30). Here are the parts you'll meet most:

```text
BaseException
 ├── KeyboardInterrupt         the user pressed Ctrl+C
 ├── SystemExit                sys.exit() was called
 └── Exception                 everything a program might want to handle
      ├── ArithmeticError
      │    └── ZeroDivisionError, OverflowError
      ├── LookupError
      │    └── IndexError, KeyError
      ├── OSError                (the operating system said no)
      │    └── FileNotFoundError, PermissionError, IsADirectoryError, TimeoutError, ConnectionError, ...
      ├── ValueError
      │    └── UnicodeError (UnicodeDecodeError, ...)
      ├── TypeError, AttributeError, NameError
      ├── RuntimeError
      │    └── RecursionError, NotImplementedError
      └── StopIteration          (Day 3 of last week)
```

You can explore it yourself, because each class knows its parents:

A few things worth knowing:

- **`OSError`** covers almost every way the outside world can fail: missing files, permissions, full discs, network timeouts. Its subclasses let you handle each precisely.
- **`UnicodeDecodeError` is a `ValueError`**, so `except ValueError` also catches text that can't be decoded. That's usually right (it is a bad value), but worth knowing.
- **`NotImplementedError` and `RecursionError` are both `RuntimeError`s**, so `except RuntimeError` catches an unfinished method and a runaway recursion alike. Be careful how widely you cast.

## Catching families

Because `except` matches subclasses (Week 30), one clause can handle a whole family, and the order of clauses matters: Python uses the **first** that matches, so put **specific** clauses before **general** ones:

If `except OSError` came first, the `FileNotFoundError` clause would never run, and nothing would warn you. One `except` can also name several unrelated types in a tuple: `except (ValueError, TypeError):`.

How general should you go? Catch **what you can actually handle** at that point. A function that reads a settings file can sensibly handle "file not found" (use defaults); it can't do anything useful about a `TypeError` caused by a bug, so it shouldn't catch it.

## try, except, else, finally

A `try` statement has up to four parts:

| Part | Runs |
|---|---|
| `try:` | always, first |
| `except ...:` | only if the `try` part raised a matching exception |
| `else:` | only if the `try` part raised **nothing** |
| `finally:` | **always**, last: after success, after a handled error, after an unhandled one, even after `return` |

- `else` keeps the `try` part **small**: only the line that might fail (`int(text)`) goes in `try`, so an error in the following lines isn't mistaken for a bad number (Week 30, Day 4's "keep `try` narrow").
- `finally` is for **clean-up** that must happen whatever goes wrong: closing a connection, releasing a lock, putting a temporary change back. Notice it ran even though `return` came first. (`with` statements do the clean-up for files automatically; Phase 5 shows how they work.)
- Don't `return` from inside `finally`: it silently throws away any exception in flight. You'll fix exactly that bug in the practice.

## Outside Exception

`KeyboardInterrupt` and `SystemExit` inherit from `BaseException` **directly**, not from `Exception`. That's deliberate: they aren't errors in the program, they're requests to stop. So `except Exception:` lets them through, and Ctrl+C still works:

A **bare** `except:` (or `except BaseException:`) catches these too, which can make a program impossible to stop and hides every bug. Week 22's linter warned about it (E722). The only good reason to catch `BaseException` is to do clean-up and then **re-raise** with a bare `raise`, and `finally` usually does that job better.

## Several errors at once

Sometimes several things fail independently: checking every row of a file, or running several jobs. Raising only the first error hides the rest. Since Python 3.11, an **`ExceptionGroup`** carries several exceptions at once, and **`except*`** handles the ones of each type:

- `ExceptionGroup(message, list_of_exceptions)` wraps them; `.exceptions` gives them back.
- Each `except*` clause receives a group of just the matching exceptions, and **more than one** clause can run.
- Plain `except ExceptionGroup:` catches the whole group at once, which is often simpler when you only want to report everything.

Use groups when collecting **all** the problems is genuinely more useful than stopping at the first, which is mostly at the boundaries where data comes in.

## Practice

## Quiz

## Daily challenge

## Before you go

Find every `except` in a program of your own and ask, for each: what family does it catch, and can this code really handle all of it? Tomorrow: designing your own exceptions well, chaining them with `raise ... from`, and translating errors between layers.